# Cross-Dataset Generalisation — Colab Runner

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save  
**Estimated time:** DistilBERT × 3 datasets × 1 seed ≈ 2–2.5 hours on T4  

Run cells top-to-bottom. Each cell prints its result before moving on.

## 0. Clone repo and set working directory

In [ ]:
import os

# ── FILL THIS IN ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/adnankhalil22/fake-news-generalization.git"
# ──────────────────────────────────────────────────────────────────────────────

REPO_DIR    = "/content/fake-news-generalization"
# The git repo is rooted at the user home dir; project files live in this subpath.
PROJECT_DIR = os.path.join(REPO_DIR, "OneDrive", "Desktop", "final research")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes")
    !git -C {REPO_DIR} pull

%cd {PROJECT_DIR}
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

## 1. Install dependencies

In [ ]:
!pip install -r requirements.txt -q

# spaCy model — only needed for masking experiments, NOT for DistilBERT training.
# Do NOT use -q flag: it breaks spaCy's version detection and causes 404 errors.
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("spaCy model installed.")
else:
    # Upgrade spaCy to latest to ensure model compatibility, then retry
    subprocess.run([sys.executable, "-m", "pip", "install", "spacy", "--upgrade", "-q"])
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

print("Installation complete.")

## 2. Verify GPU and dataset access

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU — DistilBERT training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
from src.data import get_dataset, DATASETS

print("Dataset access check:")
all_ok = True
for ds in DATASETS:
    try:
        df = get_dataset(ds, 'train')
        print(f"  OK  {ds}: n={len(df)}  fake={df.label.mean():.1%}")
    except Exception as e:
        print(f"  ERR {ds}: {e}")
        all_ok = False

if all_ok:
    print("\nAll datasets loaded. Ready to train.")
else:
    print("\nFix the errors above before continuing.")

## 3. Train DistilBERT on each dataset

One seed (42) per dataset. Each dataset takes ~40 min on T4.  
**Do not close this tab** — Colab disconnects after ~90 min of inactivity.  
If it disconnects, the Trainer saves checkpoints each epoch; you can resume.

In [ ]:
from src.train import train_distilbert

SEED = 42

for ds in DATASETS:
    print(f"\n{'='*55}")
    print(f"Training DistilBERT on {ds.upper()}  (seed={SEED})")
    print(f"{'='*55}")
    metrics = train_distilbert(ds, seed=SEED)
    print(f"  val macro-F1 : {metrics.get('eval_macro_f1', 'n/a')}")
    print(f"  val accuracy : {metrics.get('eval_accuracy', 'n/a')}")

print("\nAll three DistilBERT models trained.")

## 4. Build the 3x3 evaluation matrix

In [ ]:
from src.evaluate import build_matrix

distilbert_matrix = build_matrix('distilbert', seeds=[SEED])
print("\nMatrix saved to results/matrices/distilbert_f1_matrix.csv")

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/distilbert_f1_heatmap.png'))

## 5. Save results to Google Drive

Run this so you don't lose results if Colab resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/fake-news-results'
os.makedirs(dest, exist_ok=True)
shutil.copytree('results', os.path.join(dest, 'results'), dirs_exist_ok=True)
print(f"Results saved to Google Drive: {dest}")

## 6. Print final matrix (paste this output back to Claude)

Run this last cell and copy the full output.

In [ ]:
import pandas as pd

print("\n=== DistilBERT Macro-F1 Matrix (rows=train, cols=test) ===")
df = pd.read_csv('results/matrices/distilbert_f1_matrix.csv', index_col=0)
print(df.to_string())

print("\n=== LogReg Macro-F1 Matrix (for comparison) ===")
df2 = pd.read_csv('results/matrices/logreg_f1_matrix.csv', index_col=0)
print(df2.to_string())

print("\n--- Copy everything above this line and paste to Claude ---")